# Session 8 — Your turn: Computer vision: why images need a different network

Two jobs about convolutions: predict the shapes and parameter counts before running the network, and write the sliding-filter loop yourself.

Run the cells from the top. Replace every `...` with your own code, then run the check below it. A check prints PASS or FAIL with a hint; you can run a task as often as you like.

In [ ]:
import numpy as np
import torch
from torch import nn
from scipy.signal import correlate2d
from sklearn.datasets import load_digits

cnn = nn.Sequential(nn.Conv2d(1, 16, 3), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(16, 32, 3), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(), nn.Linear(800, 10))
print(sum(p.numel() for p in cnn.parameters()), "parameters in the CNN of the session")
from aij_check import check, close, summary

## Task 1: Predict the shapes and the parameters

Write `conv_out(size, kernel, stride=1, padding=0)` (the output size of a convolution along one side) and `conv_params(c_in, c_out, kernel)` (weights plus one bias per filter).
Use them with `pool_out = size // 2` to fill `after_conv1`, `after_pool1`, `after_conv2`, `after_pool2` (the side length of the square maps, starting from a 28 x 28 image),
and `params_conv1`, `params_conv2`. Then `params_total` adds the final linear layer: `32 * after_pool2 * after_pool2` inputs to 10 outputs, with biases.

In [ ]:
def conv_out(size, kernel, stride=1, padding=0):
    ...


def conv_params(c_in, c_out, kernel):
    ...


after_conv1 = ...
after_pool1 = ...
after_conv2 = ...
after_pool2 = ...
params_conv1 = ...
params_conv2 = ...
params_total = ...

In [ ]:
check("the side lengths are 26, 13, 11, 5", lambda: (after_conv1, after_pool1, after_conv2, after_pool2) == (26, 13, 11, 5))
check("the first layer has 160 parameters, the second 4,640", lambda: (params_conv1, params_conv2) == (160, 4640))
check("the total matches PyTorch's own count", lambda: params_total == sum(p.numel() for p in cnn.parameters()), "Do not forget the 10 biases of the linear layer.")
check("conv_out handles padding and stride", lambda: conv_out(28, 3, stride=2, padding=1) == 14)
summary()

## Task 2: Slide a filter yourself

Write `convolve(image, kernel)` that slides the kernel over the image with no padding (the 'valid' region), multiplying and adding at every position (as in the session, it does not flip the kernel).
Apply a horizontal-edge filter `[[-1,-1,-1],[0,0,0],[1,1,1]]` to the first digit of `load_digits()` and store the position of its strongest absolute response in `strongest`, as `(row, column)`.

In [ ]:
def convolve(image, kernel):
    ...


image = load_digits().images[0] / 16.0
horizontal = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype=float)
response = ...
strongest = ...
print(response.shape, strongest)

In [ ]:
check("the output is 6 x 6 for an 8 x 8 image and a 3 x 3 filter", lambda: np.shape(response) == (6, 6))
check("it agrees with scipy's correlate2d", lambda: close(response, correlate2d(image, horizontal, mode="valid"), 1e-9))
check("strongest is the position of the largest absolute response", lambda: strongest == tuple(int(i) for i in np.unravel_index(np.abs(correlate2d(image, horizontal, mode="valid")).argmax(), (6, 6))))
summary()